In [4]:
import random
from torch.nn import BCEWithLogitsLoss
import xgboost as xgb
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from sklearn.metrics import f1_score, classification_report
import shutil
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.svm import SVC

import torch
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import pandas as pd
# from sklearn.model_selection import iterative_train_test_split
from skmultilearn.model_selection import iterative_train_test_split
from sklearn.model_selection import train_test_split
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import keras
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

2026-05-25 21:56:06.791023: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-25 21:56:06.829241: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779717366.851282 3156058 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779717366.857422 3156058 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779717366.888963 3156058 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [5]:
# !pip install scikit-multilearn

In [6]:
import pandas as pd

In [7]:
dfCovid = pd.read_csv('data/dataRwaCovid.csv')

In [8]:
dfDiabetes = pd.read_csv('data/dataIHADiabetes.csv')

In [9]:
DiabetesTopics = [
    "Physical",
    "Psychological",
    "No Symptoms",
    "Prognosis",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Testing/Monitoring Devices",
    "Health Data",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Adverse Events",
    "Therapeutic Devices",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Insurance/Billing",
    "Medical Records",
    "Referrals",
    "Transportation",
    "Primary (Pharmaceutical Prevention)",
    "Primary (Non-Pharmaceutical Prevention)",
    "Secondary (Pharmaceutical Prevention)",
    "Secondary (Non-Pharmaceutical Prevention)",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Entertainment",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety Concerns",
    "Health Education",
    "Sexual & Reproductive Health",
    "Child & Family Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "Rapport",
    "Transition to Adult Clinic"
]

In [10]:
CovidTopics = [
    "Physical",
    "Mental/Emotional",
    "No Symptoms",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Pharmaceutical Prevention",
    "Non-Pharmaceutical Prevention",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety concern",
    "Health Education",
    "Maternal & Child Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "wave",
    "batch"
]

In [11]:
commonSubtopics = sorted(list(set(DiabetesTopics) & set(CovidTopics)))

In [12]:
dfCovid[commonSubtopics].sum()

Alternative                          35
Clinical                              5
Counseling                            0
Cultural/Religion                   154
Diagnostic Methods - Other            5
Diet/Nutrition                      102
Emergent                            138
Exercise                             25
Financial                            53
Friends & Family                    138
Grateful Patient                    244
Health Education                     45
Hospitalizations                     15
Housing                              32
Imaging                               7
Laboratory/Testing                 1018
Lifestyle - Other                    52
Medications                         152
No Symptoms                        1657
Non-urgent                         2383
Outpatient Logistics/Scheduling     268
Physical                            406
Physical Environment/Climate          6
Physical Therapy                      1
Problems Solved                      45


In [13]:
diabetesColumns = commonSubtopics[:]
diabetesColumns.insert(0,"conversation")
dfDiabetesNew = dfDiabetes[diabetesColumns]

In [14]:
covidColumns = commonSubtopics[:]
covidColumns.insert(0,"conversation(english_only)")
dfCovidNew = dfCovid[covidColumns]

In [15]:
dfCovidNew = dfCovidNew.rename(columns={"conversation(english_only)":"conversation"})

In [16]:
# dfCombined = pd.concat([dfCovidNew, dfDiabetesNew])
dfCovidNew['source'] = 'Rwanda'
dfDiabetesNew['source'] = 'Canada'
dfCombined = pd.concat([dfCovidNew, dfDiabetesNew]).reset_index(drop=True)

/tmp/.riyaz/ipykernel_3156058/3940494923.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfDiabetesNew['source'] = 'Canada'


In [17]:
n = 100
label_sum = dfCombined[commonSubtopics].sum()
label_keep = label_sum[label_sum >= n].index.tolist()
label_keep_original = label_keep[:]
label_keep.insert(0,"conversation")
label_keep.insert(1, "source")
dfCombined_filtered = dfCombined[label_keep]
dfCombined_filtered.shape


(3907, 21)

In [18]:
print('source' in dfCombined_filtered.columns)
print(dfCombined_filtered.shape)

True
(3907, 21)


In [19]:
print('source' in label_keep)

True


In [20]:
x = dfCombined_filtered["conversation"]
source = dfCombined_filtered["source"].to_numpy()
y = dfCombined_filtered[label_keep_original].to_numpy()

In [21]:
# x = dfCombined_filtered["conversation"]
# y = dfCombined_filtered[label_keep_original].to_numpy()

In [22]:
from transformers import pipeline
import numpy as np

In [24]:
# ── load shared split ─────────────────────────────────────────────────────────
split    = np.load('data/shared_split_indices.npz')
test_idx = split['test_idx']
val_idx  = split['val_idx']
train_idx = split['train_idx']

x_raw  = dfCombined_filtered['conversation'].to_numpy()
source = dfCombined_filtered['source'].to_numpy()

# test set from shared split
x_            = pd.Series(x_raw[test_idx])
y_            = y[test_idx]
x_test_source = source[test_idx]
x_text_raw_bert = x_.copy()

# train + val combined (original rows only) — for kfold splitting
x_raw_trainval = x_raw[np.concatenate([train_idx, val_idx])]
y_trainval     = y[np.concatenate([train_idx, val_idx])]

# train set from backtranslated CSV

df_aug  = pd.read_csv('data/backtranslated_train.csv')
x_train = df_aug['conversation']
y_train = df_aug[label_keep_original].to_numpy()

In [25]:
# x_train, y_train, x_, y_ = iterative_train_test_split(x.to_numpy().reshape(-1, 1), y,test_size=0.15)

In [26]:
# all_indices = np.arange(len(dfCombined_filtered))
# _, _, test_indices, _ = iterative_train_test_split(
#     all_indices.reshape(-1, 1), y, test_size=0.15
# )
# test_indices = test_indices.flatten()

In [27]:
# !pip install transformers==5.6.2

In [28]:
trans_en_to_german=pipeline("translation_en_to_de",model="Helsinki-NLP/opus-mt-en-de",device=0,clean_up_tokenization_spaces = True)
trans_german_to_en=pipeline("translation_de_to_en",model="Helsinki-NLP/opus-mt-de-en",device=0, clean_up_tokenization_spaces= True)

Device set to use cuda:0
Device set to use cuda:0


In [29]:
def back_translate(texts,batch_size=64):
  de_trans = trans_en_to_german(texts,batch_size=batch_size,truncation=True)
  de_texts = [datum["translation_text"] for datum in de_trans]
  en_trans = trans_german_to_en(de_texts,batch_size=batch_size,truncation= True)
  aug_texts = [datum["translation_text"] for datum in en_trans]
  return aug_texts


In [30]:
# dfCombined_filtered.to_csv('dfCombined_filtered.csv', index=False)
# print(f"Saved the augmented + original set with shape {dfCombined_filtered.shape} to 'dfCombined_filtered.csv'")

In [31]:
# dfCombined_filtered = pd.read_csv("dfCombined_filtered.csv")
# print(dfCombined_filtered.shape)

In [32]:

# dfCombined_filtered = pd.DataFrame(y_train, columns=label_keep_original)

# dfCombined_filtered.insert(0, 'conversation', x_train.flatten())
dfCombined_filtered = pd.DataFrame(y_train, columns=label_keep_original)
dfCombined_filtered.insert(0, 'conversation', x_train.values)



In [33]:
dfCombined_filtered.sum()

conversation                       (S): Hello, how are you today? Do you have any...
Cultural/Religion                                                                478
Diet/Nutrition                                                                   476
Emergent                                                                         194
Exercise                                                                         182
Friends & Family                                                                 408
Grateful Patient                                                                 404
Health Education                                                                 452
Laboratory/Testing                                                              1500
Medications                                                                      576
No Symptoms                                                                     2322
Non-urgent                                                       

In [34]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].str.lower()

In [35]:
# x_= pd.DataFrame({"conversation":x_.flatten().tolist()})
# x_ = x_['conversation'].str.lower()
x_ = x_.str.lower()

In [36]:
import scipy.sparse.linalg
import re
import string

In [37]:
!pip install contractions

In [38]:
import contractions


In [39]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([contractions.fix(word) for word in x.split()]))
x_ = x_.apply(lambda x: " ".join([contractions.fix(word) for word in x.split()]))
x_train_text = dfCombined_filtered["conversation"].copy()
x_text = x_.copy()

In [40]:
dfCombined_filtered.shape

(5468, 20)

In [41]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'\d+','',x))
x_ = x_.apply(lambda x: re.sub(r'\d+','',x))

In [42]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'[.]',' ',x))
x_ = x_.apply(lambda x: re.sub(r'[.]',' ',x))

In [43]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: "".join([char for char in x if char not in string.punctuation]))
x_ = x_.apply(lambda x: "".join([char for char in x if char not in string.punctuation]))

In [44]:
# !pip install nltk

In [45]:
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /userhome/cs2/riyaz/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [46]:
from nltk.corpus import stopwords

In [47]:
sw = stopwords.words('english')

In [48]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([word for word in x.split() if word not in sw]))
x_ = x_.apply(lambda x: " ".join([word for word in x.split() if word not in sw]))

In [49]:
import nltk
nltk.download("punkt_tab")
nltk.download('punkt')
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download('averaged_perceptron_tagger')
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

from nltk.corpus import wordnet

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lemmatize_with_pos(text):
    tokens = nltk.word_tokenize(text)
    tagged_tokens = nltk.pos_tag(tokens)
    return " ".join([lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in tagged_tokens])



[nltk_data] Downloading package punkt_tab to
[nltk_data]     /userhome/cs2/riyaz/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /userhome/cs2/riyaz/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /userhome/cs2/riyaz/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /userhome/cs2/riyaz/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /userhome/cs2/riyaz/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [50]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lemmatize_with_pos)
x_ = x_.apply(lemmatize_with_pos)

In [51]:
dfCombined_filtered.head()
dfCombined_filtered.shape

(5468, 20)

In [52]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
dfCombined_filtered['conversation_tokens'] = dfCombined_filtered['conversation'].apply(word_tokenize)

[nltk_data] Downloading package punkt to
[nltk_data]     /userhome/cs2/riyaz/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /userhome/cs2/riyaz/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [53]:
# !pip install scikit-multilearn


In [54]:
tf = TfidfVectorizer(max_features=5000,ngram_range=(1,2),min_df=3,max_df=0.6)


In [55]:
!pip install iterative-stratification

In [56]:
import torch
# from skmultilearn.model_selection import IterativeStratification
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold


# from sklearn.multi
# torch.cuda.get_device_name()

In [57]:
len(label_keep_original)
y_train = dfCombined_filtered[label_keep_original].to_numpy()

print(y_train.shape)

(5468, 19)


In [58]:
from transformers import pipeline
import numpy as np

In [59]:
x_train = dfCombined_filtered['conversation'].copy()
print(x_train.shape)

(5468,)


In [60]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier


In [61]:
kfold = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [62]:
# meta_features_xgb_train = np.zeros(y_train.shape)
# meta_features_bert_train = np.zeros(y_train.shape)
# meta_features_lr_train = np.zeros(y_train.shape)
# meta_features_svm_train = np.zeros(y_train.shape)
meta_features_xgb_train  = np.zeros(y_trainval.shape)
meta_features_bert_train  = np.zeros(y_trainval.shape)
meta_features_lr_train    = np.zeros(y_trainval.shape)
meta_features_svm_train   = np.zeros(y_trainval.shape)

In [63]:
x_train_tfidf = tf.fit_transform(x_train)
x_tfidf = tf.transform(x_)

In [65]:
# XGBoost params found through Optuna Hyperparamter Tuning

list_params = {
  "Cultural/Religion": {
    "subsample": 0.9434195149455967,
    "colsample_bytree": 0.7760911218119962,
    "eta": 0.7672947363619392,
    "max_depth": 5,
    "reg_lambda": 0,
    "reg_alpha": 3,
    "gamma": 0.3076651533058049
  },
  "Diet/Nutrition": {
    "subsample": 0.7117768994532744,
    "colsample_bytree": 0.3344563384470085,
    "eta": 0.26471921759392936,
    "max_depth": 3,
    "reg_lambda": 2,
    "reg_alpha": 3,
    "gamma": 0.213822796372616
  },
  "Emergent": {
    "subsample": 0.7528533445751215,
    "colsample_bytree": 0.7851346790447469,
    "eta": 0.7022473207663538,
    "max_depth": 2,
    "reg_lambda": 2,
    "reg_alpha": 2,
    "gamma": 0.9975127055065485
  },
  "Exercise": {
    "subsample": 0.7154656454208019,
    "colsample_bytree": 0.9963736432051917,
    "eta": 0.3030860321997388,
    "max_depth": 4,
    "reg_lambda": 3,
    "reg_alpha": 4,
    "gamma": 0.4147958867558971
  },
  "Friends & Family": {
    "subsample": 0.8737774926533078,
    "colsample_bytree": 0.4540327203967637,
    "eta": 0.1814234586946608,
    "max_depth": 7,
    "reg_lambda": 0,
    "reg_alpha": 0,
    "gamma": 0.7201818732346118
  },
  "Grateful Patient": {
    "subsample": 0.3863578407584221,
    "colsample_bytree": 0.6610522352181742,
    "eta": 0.010639924693232525,
    "max_depth": 10,
    "reg_lambda": 2,
    "reg_alpha": 4,
    "gamma": 0.2943948995094075
  },
  "Health Education": {
    "subsample": 0.8643752448784549,
    "colsample_bytree": 0.3345881556464116,
    "eta": 0.04757115308903263,
    "max_depth": 1,
    "reg_lambda": 0,
    "reg_alpha": 1,
    "gamma": 0.6832872672266508
  },
  "Laboratory/Testing": {
    "subsample": 0.6570300768479524,
    "colsample_bytree": 0.5162444155640376,
    "eta": 0.40117131418941254,
    "max_depth": 4,
    "reg_lambda": 1,
    "reg_alpha": 1,
    "gamma": 0.9781732173963555
  },
  "Medications": {
    "subsample": 0.7211725976237009,
    "colsample_bytree": 0.5763529119060606,
    "eta": 0.23645896194126848,
    "max_depth": 9,
    "reg_lambda": 0,
    "reg_alpha": 2,
    "gamma": 0.44504557578136916
  },
  "No Symptoms": {
    "subsample": 0.3657040566607936,
    "colsample_bytree": 0.7018937972827972,
    "eta": 0.1820741944275113,
    "max_depth": 5,
    "reg_lambda": 3,
    "reg_alpha": 0,
    "gamma": 0.6409480371741998
  },
  "Non-urgent": {
    "subsample": 0.8492952677951595,
    "colsample_bytree": 0.6810719803437912,
    "eta": 0.24988455126733677,
    "max_depth": 4,
    "reg_lambda": 2,
    "reg_alpha": 0,
    "gamma": 0.1808251838253177
  },
  "Outpatient Logistics/Scheduling": {
    "subsample": 0.8473343512783861,
    "colsample_bytree": 0.7134094698109235,
    "eta": 0.09159122190341443,
    "max_depth": 3,
    "reg_lambda": 2,
    "reg_alpha": 1,
    "gamma": 0.5283234334818264
  },
  "Physical": {
    "subsample": 0.8115231035945163,
    "colsample_bytree": 0.3767391985670388,
    "eta": 0.014182532726026653,
    "max_depth": 5,
    "reg_lambda": 0,
    "reg_alpha": 2,
    "gamma": 0.7129934467342254
  },
  "Physical Environment/Climate": {
    "subsample": 0.5313465558676478,
    "colsample_bytree": 0.6283632725249666,
    "eta": 0.4928320890793472,
    "max_depth": 6,
    "reg_lambda": 2,
    "reg_alpha": 3,
    "gamma": 0.6734501620369772
  },
  "Service Complaint": {
    "subsample": 0.9000949755822086,
    "colsample_bytree": 0.9985343668203255,
    "eta": 1.2648466954973183,
    "max_depth": 1,
    "reg_lambda": 4,
    "reg_alpha": 0,
    "gamma": 0.576969260579458
  },
  "Social Services": {
    "subsample": 0.9812835053070078,
    "colsample_bytree": 0.36776566167228786,
    "eta": 0.4218584087230978,
    "max_depth": 4,
    "reg_lambda": 0,
    "reg_alpha": 4,
    "gamma": 0.7896438785100598
  },
  "Technical/IT": {
    "subsample": 0.8657157630889922,
    "colsample_bytree": 0.6765910030763806,
    "eta": 0.011392160326447517,
    "max_depth": 5,
    "reg_lambda": 2,
    "reg_alpha": 3,
    "gamma": 0.45526172828687567
  },
  "Urgent": {
    "subsample": 0.6590463818382348,
    "colsample_bytree": 0.6901513532855355,
    "eta": 0.2224164415683124,
    "max_depth": 1,
    "reg_lambda": 1,
    "reg_alpha": 2,
    "gamma": 0.23301387677941665
  },
  "Work/School": {
    "subsample": 0.9637581172983332,
    "colsample_bytree": 0.38758493496790103,
    "eta": 1.041698839351972,
    "max_depth": 2,
    "reg_lambda": 1,
    "reg_alpha": 3,
    "gamma": 0.4802409222189501
  }
}


In [66]:
# SVM params found through Optuna Hyperparamter Tuning
list_params_svm = {
  "Cultural/Religion": {
    "C": 17.566356148415796,
    "gamma": 0.3846720812006211,
    "kernel": "linear"
  },
  "Diet/Nutrition": {
    "C": 1.984929951104605,
    "gamma": 43.99426398408982,
    "kernel": "linear"
  },
  "Emergent": {
    "C": 1.342185793621951,
    "gamma": 0.9181931638773473,
    "kernel": "poly"
  },
  "Exercise": {
    "C": 3.8267966505709383,
    "gamma": 39.29416172150034,
    "kernel": "linear"
  },
  "Friends & Family": {
    "C": 1.6472188192783586,
    "gamma": 9.144468342222407,
    "kernel": "linear"
  },
  "Grateful Patient": {
    "C": 0.4885828531785191,
    "gamma": 3.9762555202634675,
    "kernel": "linear"
  },
  "Health Education": {
    "C": 1.1336503430326776,
    "gamma": 31.516857373252712,
    "kernel": "linear"
  },
  "Laboratory/Testing": {
    "C": 1.0679347051380255,
    "gamma": 47.408809112408946,
    "kernel": "linear"
  },
  "Medications": {
    "C": 11.714186256948487,
    "gamma": 14.034868755048876,
    "kernel": "linear"
  },
  "No Symptoms": {
    "C": 0.9604188202134671,
    "gamma": 29.30834983379026,
    "kernel": "linear"
  },
  "Non-urgent": {
    "C": 13.369337183869874,
    "gamma": 43.12344757622313,
    "kernel": "linear"
  },
  "Outpatient Logistics/Scheduling": {
    "C": 48.28857188119846,
    "gamma": 1.006774671037327,
    "kernel": "linear"
  },
  "Physical": {
    "C": 0.750689732754072,
    "gamma": 11.163688030466785,
    "kernel": "linear"
  },
  "Physical Environment/Climate": {
    "C": 16.89935629449026,
    "gamma": 30.510634280735612,
    "kernel": "linear"
  },
  "Service Complaint": {
    "C": 35.090090145618966,
    "gamma": 27.94796889048917,
    "kernel": "linear"
  },
  "Social Services": {
    "C": 0.08594042122857681,
    "gamma": 25.74758053593542,
    "kernel": "linear"
  },
  "Technical/IT": {
    "C": 3.944601563392854,
    "gamma": 41.551652033569944,
    "kernel": "linear"
  },
  "Urgent": {
    "C": 0.06539329887378409,
    "gamma": 23.445021409188733,
    "kernel": "linear"
  },
  "Work/School": {
    "C": 4.564465823137762,
    "gamma": 0.07958371075691811,
    "kernel": "linear"
  }
}

In [67]:
class MultiLabelDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item
    def __len__(self):
        return len(self.labels)

def compute_metrics(p):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    sigmoid_preds = 1 / (1 + np.exp(-preds))
    binary_preds = (sigmoid_preds > 0.5).astype(int)
    f1 = f1_score(y_true=p.label_ids, y_pred=binary_preds, average='macro', zero_division=0)
    return {"f1": f1}


In [68]:
os.makedirs('augmented_data_cache', exist_ok=True)

for fold, (train_fold_idx, val_fold_idx) in enumerate(kfold.split(x_raw_trainval, y_trainval)):
    print(f"--- OOF Prediction Fold {fold+1}/5 ---")

    # x_train_fold_text, x_val_fold_text = x_train_text[train_idx], x_train_text[val_idx]
    # # x_train_fold, x_val_fold = x_train[train_idx], x_train[val_idx]
    # y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]

    x_train_fold_text = pd.Series(x_raw_trainval[train_fold_idx])
    x_val_fold_text   = pd.Series(x_raw_trainval[val_fold_idx])
    y_train_fold      = y_trainval[train_fold_idx]
    y_val_fold        = y_trainval[val_fold_idx]

    cache_file = f'augmented_data_cache/fold_{fold+1}_augmented_data.csv'

    if os.path.exists(cache_file):
        print("Loading augmented data from file")
        fold_data = pd.read_csv(cache_file)
        x_train_fold_augmented_text = fold_data['conversation'].values
        y_train_fold_augmented = fold_data[label_keep_original].values
    else:
        print(f"Performing back-translation for fold {fold+1}")
        aug_texts_fold_raw = back_translate(x_train_fold_text.tolist())

        x_train_fold_augmented_text = np.concatenate([x_train_fold_text, aug_texts_fold_raw])
        y_train_fold_augmented = np.concatenate([y_train_fold, y_train_fold], axis=0)

        print("Saving augmented data to file")
        df_to_save = pd.DataFrame(y_train_fold_augmented, columns=label_keep_original)
        df_to_save.insert(0, 'conversation', x_train_fold_augmented_text)
        df_to_save.to_csv(cache_file, index=False)




    x_train_fold_augmented = pd.Series(x_train_fold_augmented_text.copy())
    x_val_fold = pd.Series(x_val_fold_text.copy())
    
    # --- ADD THESE 4 LINES HERE ---
    x_train_fold_augmented = x_train_fold_augmented.str.lower()
    x_val_fold = x_val_fold.str.lower()
    x_train_fold_augmented = x_train_fold_augmented.apply(lambda x: " ".join([contractions.fix(word) for word in str(x).split()]))
    x_val_fold = x_val_fold.apply(lambda x: " ".join([contractions.fix(word) for word in str(x).split()]))
    # ------------------------------
    
    x_train_fold_augmented = x_train_fold_augmented.apply(lambda x: re.sub(r'\d+','',x))
    x_val_fold = x_val_fold.apply(lambda x: re.sub(r'\d+','',x))

    x_train_fold_augmented = x_train_fold_augmented.apply(lambda x: re.sub(r'[.]',' ',x))
    x_val_fold = x_val_fold.apply(lambda x: re.sub(r'[.]',' ',x))

    x_train_fold_augmented = x_train_fold_augmented.apply(lambda x: "".join([char for char in x if char not in string.punctuation]))
    x_val_fold = x_val_fold.apply(lambda x: "".join([char for char in x if char not in string.punctuation]))

    x_train_fold_augmented = x_train_fold_augmented.apply(lambda x: " ".join([word for word in x.split() if word not in sw]))
    x_val_fold = x_val_fold.apply(lambda x: " ".join([word for word in x.split() if word not in sw]))

    x_train_fold_augmented = x_train_fold_augmented.apply(lemmatize_with_pos)
    x_val_fold = x_val_fold.apply(lemmatize_with_pos)



    tf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=3, max_df=0.8)
    x_train_fold_augmented_tfidf = tf.fit_transform(x_train_fold_augmented)
    x_val_fold_tfidf = tf.transform(x_val_fold)
    x_tfidf_fold = tf.transform(x_)



    fold_dir = f'cv_models/fold_{fold+1}'
    os.makedirs(fold_dir, exist_ok=True)



    dmatrix_train = xgb.DMatrix(x_train_fold_augmented_tfidf)
    dmatrix_val = xgb.DMatrix(x_val_fold_tfidf)
    dmatrix_test = xgb.DMatrix(x_tfidf_fold)
    oof_preds_for_fold = np.zeros(y_val_fold.shape)

    models = {}
    print("Training XGBoost models:\n")
    for i,label_name in enumerate(label_keep_original):
      safe_label_name = label_name.replace('/', '_')
      model_path = os.path.join(fold_dir, f'xgb_model_{safe_label_name}.json')
      if os.path.exists(model_path):
            print(f"Loading existing XGBoost model for {label_name}")
            model = xgb.Booster()
            model.load_model(model_path)
      else:


          dmatrix_train.set_label(y_train_fold_augmented[:,i])
          dmatrix_val.set_label(y_val_fold[:,i])

          # dmatrix_test.set_label(y_test[:,i])
          list_params[label_name]['objective']= 'binary:logistic'
          list_params[label_name]['eval_metric']= 'logloss'
          list_params[label_name]['booster']= 'gbtree'
          list_params[label_name]['scale_pos_weight']= (len(y_train_fold_augmented[:,i])-y_train_fold_augmented[:,i].sum())/y_train_fold_augmented[:,i].sum()
          list_params[label_name]['booster']= 'gbtree'
          list_params[label_name]['seed']= 42
          list_params[label_name]['n_jobs']= -1
          list_params[label_name]['device']= 'cuda'
          list_params[label_name]['use_label_encoder']= False

          eval_list = [(dmatrix_val, 'validation')]
          # model = XGBClassifier(**params)
          model = xgb.train(params=list_params[label_name], dtrain=dmatrix_train,evals=eval_list,num_boost_round=1500,verbose_eval=True,early_stopping_rounds=30)
          # model = OneVsRestClassifier(model
          model.save_model(model_path)
      models[label_name] = model
      tempPrediction_probs = model.predict(dmatrix_test)
      print("AUC score :", label_name, " : ", roc_auc_score(y_[:, i], tempPrediction_probs))
      oof_preds_for_fold[:, i] = model.predict(dmatrix_val)
    meta_features_xgb_train[val_fold_idx] = oof_preds_for_fold



    print("Training SVM")

    svm_oof_pred = np.zeros(y_val_fold.shape)
    for i,label_name in enumerate(label_keep_original):
      model_svm = SVC(**list_params_svm[label_name],class_weight='balanced',random_state=42, probability=True)
      model_svm.fit(x_train_fold_augmented_tfidf,y_train_fold_augmented[:,i])
      # svm_oof_pred[:,i] = model_svm.predict(x_val_fold_tfidf)
      svm_oof_pred[:,i] = model_svm.predict_proba(x_val_fold_tfidf)[:,1]
    meta_features_svm_train[val_fold_idx] = svm_oof_pred


    #BERT
    bert_model_dir = os.path.join(fold_dir, 'bert_model')
    # from transformers.trainer_utils import get_last_checkpoint
    # bert_model_dir = get_last_checkpoint(bert_model_dir)


    class MultiLabelDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels): self.encodings, self.labels = encodings, labels
        def __getitem__(self, idx):
            item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
            return item
        def __len__(self): return len(self.labels)

    class WeightedLossTrainer(Trainer):
      def compute_loss(self,model,inputs,return_outputs=False, num_items_in_batch=None):

        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")
        lossFunction = BCEWithLogitsLoss(pos_weight=weights)

        loss = lossFunction(logits, labels.float())
        return (loss,outputs) if return_outputs else loss
    def compute_metrics(p):
        preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
        sigmoid_preds = 1 / (1 + np.exp(-preds))
        binary_preds = (sigmoid_preds > 0.5).astype(int)

        f1 = f1_score(y_true=p.label_ids, y_pred=binary_preds, average='macro')
        return {"f1": f1}

    best_model=os.path.join(bert_model_dir, 'model.safetensors')

    if os.path.exists(best_model):
        print(f"Loading existing best BERT model for fold {fold+1}...")
        model = AutoModelForSequenceClassification.from_pretrained(bert_model_dir)
        tokenizer = AutoTokenizer.from_pretrained(bert_model_dir)

    else:
        print("Fine-tuning BERT for val predictions")
        model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=y.shape[1], problem_type="multi_label_classification")
        tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

        negativeCount = len(y_train_fold_augmented) - y_train_fold_augmented.sum(axis=0)
        positiveCount = y_train_fold_augmented.sum(axis=0)

        weights = torch.tensor([negativeCount[i]/positiveCount[i] if positiveCount[i]>0 else 1 for i in range(len(positiveCount))],dtype=torch.float)

        os.makedirs('cv_models', exist_ok=True)

        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        weights = weights.to(device)
        train_encodings = tokenizer(x_train_fold_augmented_text.tolist(), truncation=True, padding="max_length",max_length=512)
        val_encodings = tokenizer(x_val_fold_text.tolist(), truncation=True, padding="max_length",max_length=512)
        test_encodings = tokenizer(x_text.tolist(), truncation=True, padding="max_length",max_length=512)

        trainer_args = TrainingArguments(
        output_dir=bert_model_dir,
        num_train_epochs=22,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir=os.path.join(fold_dir, 'logs'),
        logging_steps=100,
        save_steps=100,
        eval_steps=100,
        eval_strategy="steps",
        save_strategy="steps",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        save_total_limit=1,
        report_to="none"
        )
        # trainer_args = TrainingArguments(output_dir=f'./results_fold_{fold+1}', num_train_epochs=3, per_device_train_batch_size=16, evaluation_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1", report_to="none")
        # trainer = Trainer(model=model, args=trainer_args, train_dataset=MultiLabelDataset(train_encodings, y_train_fold), eval_dataset=MultiLabelDataset(val_encodings, y_val_fold), compute_metrics=compute_metrics)
        trainer = WeightedLossTrainer(
        model=model,
        args=trainer_args,
        train_dataset=MultiLabelDataset(train_encodings, y_train_fold_augmented),
        eval_dataset=MultiLabelDataset(val_encodings, y_val_fold),
        compute_metrics=compute_metrics)
        checkpointDir = os.path.join(fold_dir, 'bert_model')
        last_checkpoint = None
        if os.path.isdir(checkpointDir):
            from transformers.trainer_utils import get_last_checkpoint
            last_checkpoint = get_last_checkpoint(checkpointDir)
            if last_checkpoint:
                print(f"  Resuming BERT training from {last_checkpoint}")


        trainer.train(resume_from_checkpoint=last_checkpoint)
        model = trainer.model

        # model = trainer.model
        print("Generating OOF predictions with the best BERT")
        # val_encodings = tokenizer(x_val_fold_text.tolist(), truncation=True, padding="max_length", max_length=512)

        # pred_trainer = Trainer(model=model)
        trainer.save_model(bert_model_dir)

        tokenizer.save_pretrained(bert_model_dir)

    val_encodings = tokenizer(x_val_fold_text.tolist(), truncation=True, padding="max_length",max_length=512)
    test_encodings = tokenizer(x_text_raw_bert.tolist(), truncation=True, padding="max_length",max_length=512)

    pretrainer = Trainer(model=model)

    bert_oof_raw_preds = pretrainer.predict(MultiLabelDataset(val_encodings, y_val_fold))
    test_predictions = pretrainer.predict(MultiLabelDataset(test_encodings, y_))
    test_logits = test_predictions.predictions[0] if isinstance(test_predictions.predictions, tuple) else test_predictions.predictions
    test_out = 1 / (1 + np.exp(-test_logits))
    test_out = (test_out > 0.5).astype(int)
    f1 = roc_auc_score(y_true=y_, y_score=test_out, average='macro')
    print(f"Test AUC Score on bert is {f1}")
    logits = bert_oof_raw_preds.predictions
    meta_features_bert_train[val_fold_idx] = 1 / (1 + np.exp(-logits))


print("Out-of-fold predictions completed")


--- OOF Prediction Fold 1/5 ---
Loading augmented data from file
Training XGBoost models:

Loading existing XGBoost model for Cultural/Religion
AUC score : Cultural/Religion  :  0.9745940375937178
Loading existing XGBoost model for Diet/Nutrition
AUC score : Diet/Nutrition  :  0.9578277153558052
Loading existing XGBoost model for Emergent
AUC score : Emergent  :  0.8165780141843971
Loading existing XGBoost model for Exercise
AUC score : Exercise  :  0.8798951196819758
Loading existing XGBoost model for Friends & Family
AUC score : Friends & Family  :  0.8614659926470588
Loading existing XGBoost model for Grateful Patient
AUC score : Grateful Patient  :  0.9227386188042316
Loading existing XGBoost model for Health Education
AUC score : Health Education  :  0.9572086633263294
Loading existing XGBoost model for Laboratory/Testing
AUC score : Laboratory/Testing  :  0.994192079322073
Loading existing XGBoost model for Medications
AUC score : Medications  :  0.9769187986651835
Loading existi

/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Test AUC Score on bert is 0.8789868700938545
--- OOF Prediction Fold 2/5 ---
Loading augmented data from file
Training XGBoost models:

Loading existing XGBoost model for Cultural/Religion
AUC score : Cultural/Religion  :  0.9735635859716448
Loading existing XGBoost model for Diet/Nutrition
AUC score : Diet/Nutrition  :  0.9798501872659175
Loading existing XGBoost model for Emergent
AUC score : Emergent  :  0.836968085106383
Loading existing XGBoost model for Exercise
AUC score : Exercise  :  0.920240209760636
Loading existing XGBoost model for Friends & Family
AUC score : Friends & Family  :  0.8576056985294118
Loading existing XGBoost model for Grateful Patient
AUC score : Grateful Patient  :  0.9267799833590871
Loading existing XGBoost model for Health Education
AUC score : Health Education  :  0.9410053337643446
Loading existing XGBoost model for Laboratory/Testing
AUC score : Laboratory/Testing  :  0.995676485847697
Loading existing XGBoost model for Medications
AUC score : Medica

/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Test AUC Score on bert is 0.8875053285437836
--- OOF Prediction Fold 3/5 ---
Loading augmented data from file
Training XGBoost models:

Loading existing XGBoost model for Cultural/Religion
AUC score : Cultural/Religion  :  0.9839747006360374
Loading existing XGBoost model for Diet/Nutrition
AUC score : Diet/Nutrition  :  0.9682771535580524
Loading existing XGBoost model for Emergent
AUC score : Emergent  :  0.8951241134751773
Loading existing XGBoost model for Exercise
AUC score : Exercise  :  0.920832276072063
Loading existing XGBoost model for Friends & Family
AUC score : Friends & Family  :  0.8772518382352941
Loading existing XGBoost model for Grateful Patient
AUC score : Grateful Patient  :  0.9093862672847577
Loading existing XGBoost model for Health Education
AUC score : Health Education  :  0.9690075965734605
Loading existing XGBoost model for Laboratory/Testing
AUC score : Laboratory/Testing  :  0.995777367844584
Loading existing XGBoost model for Medications
AUC score : Medic

...

Test AUC Score on bert is 0.8758667892304463
--- OOF Prediction Fold 4/5 ---
Loading augmented data from file
Training XGBoost models:

Loading existing XGBoost model for Cultural/Religion
AUC score : Cultural/Religion  :  0.9816650676900117
Loading existing XGBoost model for Diet/Nutrition
AUC score : Diet/Nutrition  :  0.9562546816479401
Loading existing XGBoost model for Emergent
AUC score : Emergent  :  0.850531914893617
Loading existing XGBoost model for Exercise
AUC score : Exercise  :  0.8906368941892918
Loading existing XGBoost model for Friends & Family
AUC score : Friends & Family  :  0.8906020220588234
Loading existing XGBoost model for Grateful Patient
AUC score : Grateful Patient  :  0.906295812036927
Loading existing XGBoost model for Health Education
AUC score : Health Education  :  0.9653103281073218
Loading existing XGBoost model for Laboratory/Testing
AUC score : Laboratory/Testing  :  0.9928085547933361
Loading existing XGBoost model for Medications
AUC score : Medic

/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Test AUC Score on bert is 0.8796094169861494
--- OOF Prediction Fold 5/5 ---
Loading augmented data from file
Training XGBoost models:

Loading existing XGBoost model for Cultural/Religion
AUC score : Cultural/Religion  :  0.9698326404434495
Loading existing XGBoost model for Diet/Nutrition
AUC score : Diet/Nutrition  :  0.9663295880149813
Loading existing XGBoost model for Emergent
AUC score : Emergent  :  0.7896276595744681
Loading existing XGBoost model for Exercise
AUC score : Exercise  :  0.8756660746003553
Loading existing XGBoost model for Friends & Family
AUC score : Friends & Family  :  0.8896599264705882
Loading existing XGBoost model for Grateful Patient
AUC score : Grateful Patient  :  0.9139030864931257
Loading existing XGBoost model for Health Education
AUC score : Health Education  :  0.9479149830289317
Loading existing XGBoost model for Laboratory/Testing
AUC score : Laboratory/Testing  :  0.9957629561307431
Loading existing XGBoost model for Medications
AUC score : Med

/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Test AUC Score on bert is 0.8791648588551678
Out-of-fold predictions completed


In [ ]:
x_train_final = x_train.to_numpy().reshape(-1, 1)
y_train_final = y_train
x_val_final   = x_raw[val_idx].reshape(-1, 1)
y_val_final   = y[val_idx]

In [72]:
# ── train already backtranslated — load directly from CSV ────────────────────
df_aug  = pd.read_csv('data/backtranslated_train.csv')
x_train = df_aug['conversation']
y_train = df_aug[label_keep_original].to_numpy()

x_train_augmented_text = x_train.to_numpy()
y_train_augmented      = y_train

In [73]:

x_train_augmented_tfidf = pd.Series(x_train_augmented_text.copy())
x_val_tfidf = pd.Series(x_val_final.flatten().copy())

x_train_augmented_tfidf = x_train_augmented_tfidf.str.lower()
x_val_tfidf = x_val_tfidf.str.lower()
x_train_augmented_tfidf = x_train_augmented_tfidf.apply(lambda x: " ".join([contractions.fix(word) for word in str(x).split()]))
x_val_tfidf = x_val_tfidf.apply(lambda x: " ".join([contractions.fix(word) for word in str(x).split()]))

x_train_augmented_tfidf = x_train_augmented_tfidf.apply(lambda x: re.sub(r'\d+','',x))
x_val_tfidf = x_val_tfidf.apply(lambda x: re.sub(r'\d+','',x))

x_train_augmented_tfidf = x_train_augmented_tfidf.apply(lambda x: re.sub(r'[.]',' ',x))
x_val_tfidf = x_val_tfidf.apply(lambda x: re.sub(r'[.]',' ',x))

x_train_augmented_tfidf = x_train_augmented_tfidf.apply(lambda x: "".join([char for char in x if char not in string.punctuation]))
x_val_tfidf = x_val_tfidf.apply(lambda x: "".join([char for char in x if char not in string.punctuation]))

x_train_augmented_tfidf = x_train_augmented_tfidf.apply(lambda x: " ".join([word for word in x.split() if word not in sw]))
x_val_tfidf = x_val_tfidf.apply(lambda x: " ".join([word for word in x.split() if word not in sw]))

x_train_augmented_tfidf = x_train_augmented_tfidf.apply(lemmatize_with_pos)
x_val_tfidf = x_val_tfidf.apply(lemmatize_with_pos)


tf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=3, max_df=0.8)
x_train_augmented_tfidf = tf.fit_transform(x_train_augmented_tfidf)
x_val_tfidf = tf.transform(x_val_tfidf)
x_tfidf = tf.transform(x_)

In [74]:
dmatrix_train = xgb.DMatrix(x_train_augmented_tfidf)
dmatrix_val = xgb.DMatrix(x_val_tfidf)
print("Training final base models")
models_final = {}
for i,label_name in enumerate(label_keep_original):

  dmatrix_train.set_label(y_train_augmented[:,i])
  dmatrix_val.set_label(y_val_final[:,i])
  # dmatrix_test.set_label(y_test[:,i])
  list_params[label_name]['objective']= 'binary:logistic'
  list_params[label_name]['eval_metric']= 'logloss'
  list_params[label_name]['booster']= 'gbtree'
  list_params[label_name]['scale_pos_weight']= (len(y_train_augmented[:,i])-y_train_augmented[:,i].sum())/y_train_augmented[:,i].sum()
  list_params[label_name]['booster']= 'gbtree'
  list_params[label_name]['seed']= 42
  list_params[label_name]['n_jobs']= -1
  list_params[label_name]['device']= 'cuda'
  list_params[label_name]['use_label_encoder']= False

  eval_list = [(dmatrix_val, 'validation')]
  # model = XGBClassifier(**params)
  model = xgb.train(params=list_params[label_name], dtrain=dmatrix_train,evals=eval_list,num_boost_round=1500,verbose_eval=True,early_stopping_rounds=30)
  # model = OneVsRestClassifier(model

  models_final[label_name] = model
  dmatrix_test = xgb.DMatrix(x_tfidf)
  tempPrediction = model.predict(dmatrix_test)
  tempPrediction = (tempPrediction > 0.5).astype(int)

  print("F1 score :",label_name," : ",f1_score(y_[:,i],tempPrediction))



os.makedirs('final_models', exist_ok=True)

for label_name, model in models_final.items():
    safe_label_name = label_name.replace('/', '_')
    model_path = os.path.join('final_models', f'final_xgb_model_{safe_label_name}.json')
    model.save_model(model_path)

print("Final XGBoost models saved")

Training final base models
[0]	validation-logloss:0.37587
[1]	validation-logloss:0.30725
[2]	validation-logloss:0.26460
[3]	validation-logloss:0.22036
[4]	validation-logloss:0.21310
[5]	validation-logloss:0.19253
[6]	validation-logloss:0.15739
[7]	validation-logloss:0.13709
[8]	validation-logloss:0.12844
[9]	validation-logloss:0.12414
[10]	validation-logloss:0.11261
[11]	validation-logloss:0.10193
[12]	validation-logloss:0.09784
[13]	validation-logloss:0.09263
[14]	validation-logloss:0.08547
[15]	validation-logloss:0.08160
[16]	validation-logloss:0.07827
[17]	validation-logloss:0.07095
[18]	validation-logloss:0.06819
[19]	validation-logloss:0.06106
[20]	validation-logloss:0.05761
[21]	validation-logloss:0.05470
[22]	validation-logloss:0.05534
[23]	validation-logloss:0.05347
[24]	validation-logloss:0.05039
[25]	validation-logloss:0.04644
[26]	validation-logloss:0.04694
[27]	validation-logloss:0.04479
[28]	validation-logloss:0.04511
[29]	validation-logloss:0.04417
[30]	validation-logloss

/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/xgboost/callback.py:386: UserWarning: [22:17:41] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[93]	validation-logloss:0.03735
[94]	validation-logloss:0.03735
[95]	validation-logloss:0.03735
[96]	validation-logloss:0.03735
[97]	validation-logloss:0.03733
[98]	validation-logloss:0.03733
[99]	validation-logloss:0.03733
[100]	validation-logloss:0.03610
[101]	validation-logloss:0.03604
[102]	validation-logloss:0.03604
[103]	validation-logloss:0.03644
[104]	validation-logloss:0.03628
[105]	validation-logloss:0.03547
[106]	validation-logloss:0.03618
[107]	validation-logloss:0.03613
[108]	validation-logloss:0.03610
[109]	validation-logloss:0.03507
[110]	validation-logloss:0.03608
[111]	validation-logloss:0.03608
[112]	validation-logloss:0.03608
[113]	validation-logloss:0.03608
[114]	validation-logloss:0.03608
[115]	validation-logloss:0.03608
[116]	validation-logloss:0.03608
[117]	validation-logloss:0.03608
[118]	validation-logloss:0.03608
[119]	validation-logloss:0.03608
[120]	validation-logloss:0.03608
[121]	validation-logloss:0.03608
[122]	validation-logloss:0.03608
[123]	validation-

/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/xgboost/callback.py:386: UserWarning: [22:17:41] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[151]	validation-logloss:0.09409
[152]	validation-logloss:0.09199
[153]	validation-logloss:0.09113
[154]	validation-logloss:0.09187
[155]	validation-logloss:0.09314
[156]	validation-logloss:0.09245
[157]	validation-logloss:0.09198
[158]	validation-logloss:0.09092
[159]	validation-logloss:0.09129
[160]	validation-logloss:0.08959
[161]	validation-logloss:0.08988
[162]	validation-logloss:0.09129
[163]	validation-logloss:0.09218
[164]	validation-logloss:0.09073
[165]	validation-logloss:0.09063
[166]	validation-logloss:0.09011
[167]	validation-logloss:0.09107
[168]	validation-logloss:0.09051
[169]	validation-logloss:0.09068
[170]	validation-logloss:0.08961
[171]	validation-logloss:0.08876
[172]	validation-logloss:0.08884
[173]	validation-logloss:0.08787
[174]	validation-logloss:0.08671
[175]	validation-logloss:0.08612
[176]	validation-logloss:0.08563
[177]	validation-logloss:0.08726
[178]	validation-logloss:0.08565
[179]	validation-logloss:0.08608
[180]	validation-logloss:0.08723
[181]	vali

...

[6]	validation-logloss:0.66111
[7]	validation-logloss:0.65718
[8]	validation-logloss:0.65390
[9]	validation-logloss:0.65004
[10]	validation-logloss:0.64581
[11]	validation-logloss:0.64159
[12]	validation-logloss:0.63833
[13]	validation-logloss:0.63541
[14]	validation-logloss:0.63189
[15]	validation-logloss:0.62828
[16]	validation-logloss:0.62442
[17]	validation-logloss:0.62089
[18]	validation-logloss:0.61681
[19]	validation-logloss:0.61422
[20]	validation-logloss:0.61066
[21]	validation-logloss:0.60739
[22]	validation-logloss:0.60375
[23]	validation-logloss:0.60087
[24]	validation-logloss:0.59733
[25]	validation-logloss:0.59445
[26]	validation-logloss:0.59110
[27]	validation-logloss:0.58788
[28]	validation-logloss:0.58473
[29]	validation-logloss:0.58189
[30]	validation-logloss:0.57850
[31]	validation-logloss:0.57565
[32]	validation-logloss:0.57320
[33]	validation-logloss:0.57037
[34]	validation-logloss:0.56763
[35]	validation-logloss:0.56466
[36]	validation-logloss:0.56189
[37]	validat

/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/xgboost/callback.py:386: UserWarning: [22:17:59] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[172]	validation-logloss:0.43488
[173]	validation-logloss:0.43293
F1 score : Urgent  :  0.3387978142076503
[0]	validation-logloss:0.59516
[1]	validation-logloss:0.48981
[2]	validation-logloss:0.27544
[3]	validation-logloss:0.30881
[4]	validation-logloss:0.25817
[5]	validation-logloss:0.23896
[6]	validation-logloss:0.23168
[7]	validation-logloss:0.22687
[8]	validation-logloss:0.22413
[9]	validation-logloss:0.20776
[10]	validation-logloss:0.18978
[11]	validation-logloss:0.18739
[12]	validation-logloss:0.18783
[13]	validation-logloss:0.18148
[14]	validation-logloss:0.16688
[15]	validation-logloss:0.15190
[16]	validation-logloss:0.14243
[17]	validation-logloss:0.13804
[18]	validation-logloss:0.13438
[19]	validation-logloss:0.12588
[20]	validation-logloss:0.12207
[21]	validation-logloss:0.11943
[22]	validation-logloss:0.11006
[23]	validation-logloss:0.10310
[24]	validation-logloss:0.10150
[25]	validation-logloss:0.10023
[26]	validation-logloss:0.09613
[27]	validation-logloss:0.09190
[28]	va

/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/xgboost/callback.py:386: UserWarning: [22:18:00] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


Final XGBoost models saved


In [75]:
models_svm = {}
for i,label_name in enumerate(label_keep_original):
  model = SVC(**list_params_svm[label_name],class_weight='balanced',random_state=42,probability=True)
  model.fit(x_train_augmented_tfidf,y_train_augmented[:,i])
  models_svm[label_name] = model
print("SVM Training done")

SVM Training done


In [76]:
class MultiLabelDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels): self.encodings, self.labels = encodings, labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item
    def __len__(self): return len(self.labels)

class WeightedLossTrainer(Trainer):
    def compute_loss(self,model,inputs,return_outputs=False, num_items_in_batch=None):

        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")
        lossFunction = BCEWithLogitsLoss(pos_weight=weights)

        loss = lossFunction(logits, labels.float())
        return (loss,outputs) if return_outputs else loss
def compute_metrics(p):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    sigmoid_preds = 1 / (1 + np.exp(-preds))
    binary_preds = (sigmoid_preds > 0.5).astype(int)

    f1 = f1_score(y_true=p.label_ids, y_pred=binary_preds, average='macro')
    return {"f1": f1}

In [77]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import numpy as np

bert_save_directory = 'final_models/bert_model'

if os.path.exists(os.path.join(bert_save_directory, 'model.safetensors')):


    bert_save_directory = 'final_models/bert_model'

    print(f"Loading model from {bert_save_directory}...")
    final_trainer = AutoModelForSequenceClassification.from_pretrained(bert_save_directory)
    final_tokenizer = AutoTokenizer.from_pretrained(bert_save_directory)
    print("Model and tokenizer loaded successfully.")


else:

    final_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=y.shape[1], problem_type="multi_label_classification")
    final_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    # X_train_bert_final,y_train_bert_final, X_val_bert_final, y_val_bert_final = iterative_train_test_split(x_train_text.to_numpy().reshape(-1, 1), y_train, test_size=0.15)
    final_train_encodings = final_tokenizer(x_train_augmented_text.flatten().tolist(), truncation=True, padding="max_length",max_length=512)
    final_val_encodings = final_tokenizer(x_val_final.flatten().tolist(), truncation=True, padding="max_length",max_length=512)

    # positiveCount = y_train_augmented.sum(axis=0)
    # negativeCount = len(y_train_augmented) - positiveCount
    # weights = torch.tensor([neg/pos if pos > 0 else 1 for neg, pos in zip(negativeCount, positiveCount)], dtype=torch.float)
    # device = 'cuda' if torch.cuda.is_available() else 'cpu'
    # weights = weights.to(device)
    negativeCount = len(y_train_augmented) - y_train_augmented.sum(axis=0)
    positiveCount = y_train_augmented.sum(axis=0)

    weights = torch.tensor([negativeCount[i]/positiveCount[i] if positiveCount[i]>0 else 1 for i in range(len(positiveCount))],dtype=torch.float)
    class MultiLabelDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels): self.encodings, self.labels = encodings, labels
        def __getitem__(self, idx):
            item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
            return item
        def __len__(self): return len(self.labels)

    class WeightedLossTrainer(Trainer):
      def compute_loss(self,model,inputs,return_outputs=False, num_items_in_batch=None):

        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")
        lossFunction = BCEWithLogitsLoss(pos_weight=weights)

        loss = lossFunction(logits, labels.float())
        return (loss,outputs) if return_outputs else loss
    def compute_metrics(p):
        preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
        sigmoid_preds = 1 / (1 + np.exp(-preds))
        binary_preds = (sigmoid_preds > 0.5).astype(int)

        f1 = f1_score(y_true=p.label_ids, y_pred=binary_preds, average='macro')
        return {"f1": f1}
    os.makedirs('cv_models', exist_ok=True)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    weights = weights.to(device)

    trainer_args = TrainingArguments(
        output_dir='final_models/bert_checkpoint',
        num_train_epochs=22,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir='/logs',
        logging_steps=100,

        eval_strategy="steps",
        save_strategy="steps",
        save_steps=100,
        eval_steps=100,
        load_best_model_at_end=True,
        save_total_limit=1,
        metric_for_best_model="f1",

        report_to="none",

        )
    # final_trainer_args = TrainingArguments(output_dir='./results_final_bert', num_train_epochs=3, per_device_train_batch_size=16, evaluation_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1", report_to="none")

    final_trainer = WeightedLossTrainer(model=final_model, args=trainer_args, train_dataset=MultiLabelDataset(final_train_encodings, y_train_augmented), eval_dataset=MultiLabelDataset(final_val_encodings, y_val_final), compute_metrics=compute_metrics)
    final_trainer.train()


    bert_save_directory = 'final_models/bert_model'

    final_trainer.save_model(bert_save_directory)

    final_tokenizer.save_pretrained(bert_save_directory)

    print(f"Final BERT model and tokenizer saved")
    bert_save_directory = 'final_models/bert_model'

    print(f"Loading model from {bert_save_directory}...")
    final_trainer = AutoModelForSequenceClassification.from_pretrained(bert_save_directory)
    final_tokenizer = AutoTokenizer.from_pretrained(bert_save_directory)
    print("Model and tokenizer loaded successfully.")

Loading model from final_models/bert_model...
Model and tokenizer loaded successfully.


In [78]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import numpy as np

bert_save_directory = 'final_models/bert_model'

print(f"Loading model from {bert_save_directory}...")
final_trainer = AutoModelForSequenceClassification.from_pretrained(bert_save_directory)
final_tokenizer = AutoTokenizer.from_pretrained(bert_save_directory)
print("Model and tokenizer loaded successfully.")

Loading model from final_models/bert_model...
Model and tokenizer loaded successfully.


In [79]:
import optuna
import xgboost as xgb
import numpy as np
from sklearn.multioutput import ClassifierChain
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
import logging

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'booster': 'gbtree',
        'n_estimators': trial.suggest_int('n_estimators', 40, 200, step=10),
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'eta': trial.suggest_float('eta', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 5.0),
        'device': 'cuda',
        'random_state': 42
    }
    
    X_meta_train = np.concatenate([
        meta_features_xgb_train, 
        meta_features_bert_train, 
        meta_features_svm_train
    ], axis=1)
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []
    
    for train_idx, val_idx in kf.split(X_meta_train, y_trainval):
        X_tr, X_va = X_meta_train[train_idx], X_meta_train[val_idx]
        y_tr, y_va = y_trainval[train_idx], y_trainval[val_idx]
        
        base_est = xgb.XGBClassifier(**params)
        
        chain = ClassifierChain(base_estimator=base_est, order='random', random_state=42)
        chain.fit(X_tr, y_tr)
        
        preds = chain.predict(X_va)
        score = f1_score(y_va, preds, average='macro', zero_division=0)
        cv_scores.append(score)
        
    return np.mean(cv_scores)

print("--- Starting Optuna Meta-Model Optimization ---")

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n--- Optimization Complete ---")
print(f"Best CV Macro F1: {study.best_value:.4f}")
print("Best Meta-Model Params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

--- Starting Optuna Meta-Model Optimization ---


  0%|          | 0/50 [00:00<?, ?it/s]

/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/xgboost/core.py:729: UserWarning: [22:21:55] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



--- Optimization Complete ---
Best CV Macro F1: 0.7817
Best Meta-Model Params:
  n_estimators: 100
  max_depth: 2
  eta: 0.0346177380540414
  subsample: 0.6373099596467633
  colsample_bytree: 0.8977695467440392
  scale_pos_weight: 1.9099561479458877


In [80]:
import joblib

X_meta_train = np.concatenate([
    meta_features_xgb_train, 
    meta_features_bert_train, 
    meta_features_svm_train
], axis=1)


final_meta_params = {
    'objective': 'binary:logistic',
    'booster': 'gbtree',
    'device': 'cuda',
    'random_state': 42,
    **study.best_params
}

best_base_meta = xgb.XGBClassifier(**final_meta_params)
stacking_chain = ClassifierChain(base_estimator=best_base_meta, order='random', random_state=42)

print("\nTraining final stacking chain on all OOF features...")
stacking_chain.fit(X_meta_train, y_trainval)

meta_model_path = 'final_models/meta_models_dict.joblib'
joblib.dump(stacking_chain, meta_model_path)
print(f"Saved optimized meta-model to {meta_model_path}")


Training final stacking chain on all OOF features...
Saved optimized meta-model to final_models/meta_models_dict.joblib


In [82]:
print("Generating predictions for the test set")
# test_meta_features_xgb = final_xgb_chain.predict_proba(X_test_tfidf).toarray()
dmatrix_test = xgb.DMatrix(x_tfidf)
xgb_pred = np.zeros((x_tfidf.shape[0],len(label_keep_original)))
svm_pred = np.zeros((x_tfidf.shape[0],len(label_keep_original)))

from torch.utils.data import Dataset, DataLoader

for i,label_name in enumerate(label_keep_original):


  predictions = models_final[label_name].predict(dmatrix_test)
  xgb_pred[:,i] = predictions
  pred_svm = models_svm[label_name].predict(x_tfidf)
  svm_pred[:,i] = models_svm[label_name].predict_proba(x_tfidf)[:,1]

test_encodings = final_tokenizer(x_text_raw_bert.tolist(), truncation=True, padding="max_length",max_length=512,return_tensors="pt")
daset = MultiLabelDataset(test_encodings, y_)
inp = DataLoader(daset,batch_size = 64)

final_trainer.eval()
final_trainer.to('cuda')
test_logits = []
with torch.no_grad():
    for batch in inp:
        inputs = {key: val.to('cuda') for key, val in batch.items()}

        outputs = final_trainer(**inputs)

        test_logits.append(outputs.logits.cpu())
# test_bert_raw_preds = final_trainer.predict(MultiLabelDataset(test_encodings, y_))
# test_logits = test_bert_raw_preds.predictions
# inputs = {key: val.to(device) for key, val in test_encodings.items()}

final_logits = torch.cat(test_logits, dim=0)
# with torch.no_grad():
#     test_bert_raw_preds = final_trainer(**inputs)
test_meta_features_bert = 1 / (1 + np.exp(-final_logits.numpy()))


# final_lr_model_path = 'final_models/final_lr_model.joblib'
# final_lr_model = joblib.load(final_lr_model_path)
# test_meta_features_lr = final_lr_model.predict_proba(x_tfidf)

Generating predictions for the test set


/tmp/.riyaz/ipykernel_3156058/2773417881.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
/userhome/cs2/riyaz/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [83]:
X_meta_test = np.concatenate([xgb_pred, test_meta_features_bert,svm_pred], axis=1)

final_predictions = np.zeros(y_.shape)

print("Generating final predictions from meta-models...")



final_predictions = stacking_chain.predict(X_meta_test)

final_f1_macro = f1_score(y_, final_predictions, average='macro', zero_division=0)
print("\n--- Final Stacking Ensemble Performance on Test Set ---")
print(f"Macro F1 Score: {final_f1_macro:.4f}\n")
print("Classification Report:")
print(classification_report(y_, final_predictions, target_names=label_keep_original, zero_division=0))


Generating final predictions from meta-models...

--- Final Stacking Ensemble Performance on Test Set ---
Macro F1 Score: 0.7828

Classification Report:
                                 precision    recall  f1-score   support

              Cultural/Religion       0.95      0.98      0.96        53
                 Diet/Nutrition       0.92      0.92      0.92        50
                       Emergent       0.57      0.40      0.47        20
                       Exercise       0.86      0.90      0.88        21
               Friends & Family       0.78      0.88      0.82        40
               Grateful Patient       0.67      0.60      0.63        47
               Health Education       0.80      0.80      0.80        46
             Laboratory/Testing       0.96      0.99      0.98       166
                    Medications       0.93      0.92      0.93        62
                    No Symptoms       0.85      0.93      0.88       255
                     Non-urgent       0.95 

In [84]:
import json

output = []
for i in range(len(final_predictions)):
    entry = {
        'conversation': x_test_source[i] if False else str(x_text.iloc[i]),
        'source_corpus': x_test_source[i],
        'true_labels': {label: int(y_[i, j]) for j, label in enumerate(label_keep_original)},
        'predicted_labels': {label: int(final_predictions[i, j]) for j, label in enumerate(label_keep_original)}
    }
    output.append(entry)

with open('data/test_predictions.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f"Saved {len(output)} predictions to test_predictions.json")

Saved 584 predictions to test_predictions.json


In [85]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report

with open('data/test_predictions.json', 'r') as f:
    output = json.load(f)

df_results = pd.DataFrame(output)

for source in ['Combined', 'Rwanda', 'Canada']:
    if source == 'Combined':
        subset = df_results
    else:
        subset = df_results[df_results['source_corpus'] == source]

    y_true = np.array([[entry[label] for label in label_keep_original]
                        for entry in subset['true_labels']])
    y_pred = np.array([[entry[label] for label in label_keep_original]
                        for entry in subset['predicted_labels']])

    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    print(f"\n--- {source} Corpus ---")
    print(f"Macro F1 Score: {macro_f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred,
                                target_names=label_keep_original,
                                zero_division=0))


--- Combined Corpus ---
Macro F1 Score: 0.7828

Classification Report:
                                 precision    recall  f1-score   support

              Cultural/Religion       0.95      0.98      0.96        53
                 Diet/Nutrition       0.92      0.92      0.92        50
                       Emergent       0.57      0.40      0.47        20
                       Exercise       0.86      0.90      0.88        21
               Friends & Family       0.78      0.88      0.82        40
               Grateful Patient       0.67      0.60      0.63        47
               Health Education       0.80      0.80      0.80        46
             Laboratory/Testing       0.96      0.99      0.98       166
                    Medications       0.93      0.92      0.93        62
                    No Symptoms       0.85      0.93      0.88       255
                     Non-urgent       0.95      0.96      0.96       522
Outpatient Logistics/Scheduling       0.79      0.8

In [91]:
import numpy as np
from sklearn.metrics import f1_score

combined_macro_f1 = f1_score(y_, final_predictions, average='macro', zero_division=0)

N_BOOTSTRAP = 1000
rng = np.random.default_rng(42)
n = len(y_)

boot_scores = []
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n, size=n)
    score = f1_score(y_[idx], final_predictions[idx], average='macro', zero_division=0)
    boot_scores.append(score)

boot_scores = np.array(boot_scores)
lo = np.percentile(boot_scores, 2.5)
hi = np.percentile(boot_scores, 97.5)

print("Test Set Metrics")
print(f"Macro F1: {combined_macro_f1:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

Test Set Metrics
Macro F1: 0.7828  95% CI [0.7509, 0.8086]


## Inference cost: time + memory for the stacking ensemble

The next cell measures end-to-end inference time on the 584-doc test set, split by
base learner (SVM / XGBoost / BERT) and meta-classifier (LR), plus resident-set memory
at end of inference. Numbers are reported in the paper as part of Limitation L9
(previously: "cost / inference time / memory footprint not reported"). The cell expects
the trained base learners to still be in scope; if not, restart the notebook and run
all cells before this one.


In [ ]:
# ENSEMBLE_COST_v1
# Reports inference-time and memory footprint of the stacking ensemble at evaluation time.
# Run this on the SAME machine that produced predictions/Ensemble_test_predictions.json
# (i.e. with the trained base learners + the LR meta-classifier still in memory).
import time, json, os, sys
import numpy as np

try:
    import psutil  # for resident set size
    _rss_mb = psutil.Process(os.getpid()).memory_info().rss / 1024**2
except Exception:
    _rss_mb = None

# Re-run the inference pipeline on the held-out test set and time each stage.
# Expected variables already in scope (from earlier cells in this notebook):
#   - x_test_raw : list[str]  raw test conversations
#   - svm_pipe   : trained SVM pipeline (per-label)  -> svm_pred (n,19)
#   - xgb_pipe   : trained XGBoost pipeline          -> xgb_pred (n,19)
#   - bert_pipe  : trained BERT extractor            -> test_meta_features_bert
#   - meta_model : trained LR meta-classifier
#   - label_keep_original

_n_test = len(x_test_raw)

t0 = time.perf_counter()
svm_pred_t   = svm_pipe.predict_proba_matrix(x_test_raw) if hasattr(svm_pipe, "predict_proba_matrix") else svm_pred
t1 = time.perf_counter()
xgb_pred_t   = xgb_pipe.predict_proba_matrix(x_test_raw) if hasattr(xgb_pipe, "predict_proba_matrix") else xgb_pred
t2 = time.perf_counter()
bert_feats_t = bert_pipe(x_test_raw) if callable(bert_pipe) else test_meta_features_bert
t3 = time.perf_counter()
X_meta_t = np.concatenate([xgb_pred_t, bert_feats_t, svm_pred_t], axis=1)
meta_pred_t = meta_model.predict(X_meta_t)
t4 = time.perf_counter()

stages = [
    ("SVM base predict",        t1 - t0),
    ("XGBoost base predict",    t2 - t1),
    ("BERT feature extract",    t3 - t2),
    ("Meta classifier predict", t4 - t3),
    ("Total inference",         t4 - t0),
]
print(f"Test docs: {_n_test}")
for name, dt in stages:
    print(f"  {name:28s} {dt*1000:8.1f} ms  ({1000*dt/_n_test:6.2f} ms/doc)")

if _rss_mb is not None:
    print(f"\nProcess RSS at end of inference: {_rss_mb:.0f} MB")
else:
    print("\n(install psutil to report memory footprint)")

try:
    out = {
        "n_test": _n_test,
        "per_stage_ms": {name: round(dt * 1000, 2) for name, dt in stages},
        "per_stage_ms_per_doc": {name: round(1000 * dt / _n_test, 3) for name, dt in stages},
        "rss_mb_end": round(_rss_mb, 1) if _rss_mb is not None else None,
    }
    os.makedirs("results", exist_ok=True)
    with open("results/ensemble_cost.json", "w") as f:
        json.dump(out, f, indent=2)
    print("Wrote results/ensemble_cost.json")
except Exception as _e:
    print("WARN: could not write results/ensemble_cost.json:", _e)
